In [1]:
import boto3
import json
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime
import os
from dotenv import load_dotenv
import re
import uuid

# Load env file
load_dotenv()

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name="ap-south-1",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY
)

In [2]:
class RCAOutput(BaseModel):
    incident_id: str = Field(description="Unique ID for the incident")
    severity: str = Field(description="Critical, High, Medium, or Low")
    summary: str = Field(description="One-sentence summary of what happened")
    root_cause: str = Field(description="The primary technical reason for the failure")
    immediate_fix: str = Field(description="What to do right now to restore service")
    confidence_score: float = Field(description="AI's confidence (0.0 to 1.0)")
    model_used: str = Field(description="Llama-3-8B or Llama-3-70B")

json_schema = RCAOutput.model_json_schema()

In [3]:
def generate_infra_rca(raw_log: str, context: str = "") -> RCAOutput:

    # 0. Generate a unique incident ID for tracking
    incident_id = str(uuid.uuid4())

    # 1. Simple Routing Logic
    # If the log is massive, we escalate to the 70B model
    selected_model = "meta.llama3-70b-instruct-v1:0" if len(raw_log) > 2000 else "meta.llama3-8b-instruct-v1:0"
    
    # 2. Construct the Prompt
    prompt = f"""
    <system>
    You are a Senior SRE. 
    The incident ID is: {incident_id}
    Analyze the log and return ONLY valid JSON. Do not include explanations, markdown, or text outside JSON.
    Format your response strictly according to this JSON schema: {json.dumps(json_schema)}
    You MUST include the provided incident_id in the JSON.
    </system>
    
    <incident_context>
    {context}
    </incident_context>
    
    <log>
    {raw_log}
    </log>
    
    JSON Output:
    """

    # 3. Call AWS Bedrock
    response = bedrock_runtime.invoke_model(
        modelId=selected_model,
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "prompt": prompt,
            "max_gen_len": 1024,
            "temperature": 0.1,
            "top_p": 0.9,
            "stop": ["\n\n"]
        })
    )
    
    # 4. Parse and Validate
    result = json.loads(response["body"].read())
    generation = result["generation"]

    # Extract JSON block
    match = re.search(r"\{.*\}", generation, re.DOTALL)

    if not match:
        raise ValueError("No JSON found in model output")

    json_text = match.group()

    try:
        rca = RCAOutput.model_validate_json(json_text)
    except Exception as e:
        raise ValueError(f"Model returned invalid JSON: {json_text}") from e

    # Force correct metadata
    rca.incident_id = incident_id
    model_label = "Llama-3-70B" if "70b" in selected_model else "Llama-3-8B"
    rca.model_used = model_label

    return rca

In [4]:
test_log = "ERROR: [Kubelet] Failed to pull image 'nginx:latest': RPC error: code = Unknown desc = Error response from daemon: Get https://registry-1.docker.io/v2/: net/http: TLS handshake timeout"

try:
    analysis = generate_infra_rca(test_log, context="VPC Security Groups allow outbound 443.")
    print("\n--- RCA REPORT ---")
    print("Incident ID:", analysis.incident_id)
    print("Severity:", analysis.severity)
    print("Summary:", analysis.summary)
    print("Root Cause:", analysis.root_cause)
    print("Immediate Fix:", analysis.immediate_fix)
    print("Confidence:", analysis.confidence_score)
    print("Model:", analysis.model_used)
except Exception as e:
    print(f"Mission Failed: {e}")


--- RCA REPORT ---
Incident ID: 0cf8b56c-f2e4-45eb-a5e6-01662526e92d
Severity: High
Summary: Failed to pull image 'nginx:latest' due to TLS handshake timeout
Root Cause: TLS handshake timeout with Docker registry
Immediate Fix: Check Docker registry connectivity and try again
Confidence: 0.8
Model: Llama-3-8B
